# Deep Past: Akkadian-English Translation - Kaggle Inference Notebook

This notebook generates predictions for the competition test set using a trained model.

## Instructions
1. Make sure you have trained a model using `kaggle_training.ipynb` first
2. Or upload a pre-trained model as a Kaggle dataset
3. Add the competition test data
4. Run all cells to generate `submission.csv`

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q transformers==4.40.0 sentencepiece

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# CONFIGURATION - MODIFY THESE FOR YOUR SETUP
# ============================================================================

# Kaggle dataset names
MODEL_DATASET_NAME = "deep-past-akkadian"  # Dataset containing trained model
COMPETITION_DATASET_NAME = "deep-past"  # Competition dataset name

# Check environment
RUNNING_ON_KAGGLE = os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if RUNNING_ON_KAGGLE:
    INPUT_BASE = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    
    # Find model checkpoint
    possible_model_paths = [
        WORKING_DIR / "models" / "submission_model",  # From training notebook
        INPUT_BASE / MODEL_DATASET_NAME / "models" / "submission_model",
        INPUT_BASE / MODEL_DATASET_NAME / "submission_model",
        INPUT_BASE / "akkadian-model" / "submission_model",
    ]
    
    MODEL_DIR = None
    for path in possible_model_paths:
        if path.exists() and (path / "config.json").exists():
            MODEL_DIR = path
            break
    
    # Find test data
    possible_test_paths = [
        INPUT_BASE / COMPETITION_DATASET_NAME / "test.csv",
        INPUT_BASE / MODEL_DATASET_NAME / "data" / "competition" / "test.csv",
        INPUT_BASE / "deep-past" / "test.csv",
    ]
    
    TEST_CSV = None
    for path in possible_test_paths:
        if path.exists():
            TEST_CSV = path
            break
    
    OUTPUT_DIR = WORKING_DIR
else:
    # Local paths
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "kaggle" else Path.cwd()
    MODEL_DIR = PROJECT_ROOT / "models" / "submission_model"
    TEST_CSV = PROJECT_ROOT / "data" / "competition" / "test.csv"
    OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running on Kaggle: {RUNNING_ON_KAGGLE}")
print(f"Model directory: {MODEL_DIR}")
print(f"Test CSV: {TEST_CSV}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Validate paths
if MODEL_DIR is None or not MODEL_DIR.exists():
    print("[ERROR] Model directory not found!")
    print("\nPlease either:")
    print("1. Run the training notebook first")
    print("2. Upload a trained model as a Kaggle dataset")
    
    if RUNNING_ON_KAGGLE:
        print("\nAvailable datasets:")
        for p in Path("/kaggle/input").iterdir():
            print(f"  - {p.name}")
    raise FileNotFoundError("Model not found")

if TEST_CSV is None or not TEST_CSV.exists():
    print("[ERROR] Test CSV not found!")
    print("Please add the competition dataset to your notebook.")
    raise FileNotFoundError("Test data not found")

print("[OK] All required files found!")

## 2. Inference Configuration

In [ ]:
# Inference parameters
BATCH_SIZE = 16  # Can be larger for inference (no gradients)
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 512
NUM_BEAMS = 4
EARLY_STOPPING = True

# mBART language codes
SRC_LANG = "ar_AR"  # Arabic as proxy for Akkadian
TGT_LANG = "en_XX"  # English

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Load Model

In [ ]:
print(f"Loading model from: {MODEL_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

# Configure tokenizer
if hasattr(tokenizer, "src_lang"):
    tokenizer.src_lang = SRC_LANG
if hasattr(tokenizer, "tgt_lang"):
    tokenizer.tgt_lang = TGT_LANG

model.to(device)
model.eval()

print("Model loaded successfully!")

## 4. Load Test Data

In [ ]:
print(f"Loading test data from: {TEST_CSV}")

test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")
print(f"Columns: {test_df.columns.tolist()}")

# Determine source column
if "transliteration" in test_df.columns:
    SRC_COL = "transliteration"
elif "transliteration_normalized" in test_df.columns:
    SRC_COL = "transliteration_normalized"
else:
    raise ValueError(f"Cannot find source column. Available: {test_df.columns.tolist()}")

# Check for ID column
if "id" in test_df.columns:
    ID_COL = "id"
elif "ID" in test_df.columns:
    ID_COL = "ID"
else:
    # Create ID column
    test_df["id"] = range(1, len(test_df) + 1)
    ID_COL = "id"

print(f"Source column: {SRC_COL}")
print(f"ID column: {ID_COL}")
print(f"\nSample data:")
test_df[[ID_COL, SRC_COL]].head()

## 5. Run Inference

In [ ]:
def run_batched_inference(sources, batch_size=16):
    """
    Run inference on a list of source texts in batches.
    
    Args:
        sources: List of source texts
        batch_size: Number of samples per batch
    
    Returns:
        List of translations
    """
    translations = []
    total_batches = (len(sources) + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, len(sources), batch_size), total=total_batches, desc="Translating"):
        batch_sources = sources[i:i + batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_sources,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=NUM_BEAMS,
                early_stopping=EARLY_STOPPING,
            )
        
        # Decode
        batch_translations = tokenizer.batch_decode(
            outputs, skip_special_tokens=True
        )
        translations.extend(batch_translations)
    
    return translations

In [ ]:
# Get source texts
sources = test_df[SRC_COL].tolist()
ids = test_df[ID_COL].tolist()

print(f"Running inference on {len(sources)} samples...")

# Run inference
translations = run_batched_inference(sources, batch_size=BATCH_SIZE)

print(f"\nGenerated {len(translations)} translations")

In [ ]:
# Preview results
print("Sample translations:\n")
for i in range(min(5, len(translations))):
    print(f"ID: {ids[i]}")
    print(f"Source: {sources[i][:100]}..." if len(sources[i]) > 100 else f"Source: {sources[i]}")
    print(f"Translation: {translations[i]}")
    print("-" * 50)

## 6. Create Submission File

In [ ]:
# Create submission DataFrame
submission_df = pd.DataFrame({
    "id": ids,
    "translation": translations,
})

# Save submission
submission_path = OUTPUT_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total rows: {len(submission_df)}")
print(f"\nFirst few rows:")
submission_df.head()

In [ ]:
# Validate submission
def validate_submission(submission_df, test_df):
    """Validate submission format."""
    errors = []
    
    # Check columns
    expected_cols = {"id", "translation"}
    actual_cols = set(submission_df.columns)
    if actual_cols != expected_cols:
        errors.append(f"Wrong columns: {actual_cols}, expected {expected_cols}")
    
    # Check row count
    if len(submission_df) != len(test_df):
        errors.append(f"Wrong row count: {len(submission_df)}, expected {len(test_df)}")
    
    # Check for missing IDs
    test_ids = set(test_df[ID_COL].tolist())
    submission_ids = set(submission_df["id"].tolist())
    missing = test_ids - submission_ids
    if missing:
        errors.append(f"Missing IDs: {list(missing)[:5]}...")
    
    # Check for duplicate IDs
    duplicates = submission_df[submission_df["id"].duplicated()]["id"].tolist()
    if duplicates:
        errors.append(f"Duplicate IDs: {duplicates[:5]}...")
    
    # Check for empty translations
    empty = submission_df[submission_df["translation"].isna() | (submission_df["translation"] == "")]
    if len(empty) > 0:
        errors.append(f"Empty translations: {len(empty)} rows")
    
    return errors

# Run validation
errors = validate_submission(submission_df, test_df)

if errors:
    print("[VALIDATION FAILED]")
    for err in errors:
        print(f"  - {err}")
else:
    print("[VALIDATION PASSED]")
    print("Your submission is ready for upload!")

## 7. (Optional) Apply Glossary Post-Processing

In [ ]:
# Load glossary if available
GLOSSARY_PATH = None

if RUNNING_ON_KAGGLE:
    possible_glossary_paths = [
        Path("/kaggle/input") / MODEL_DATASET_NAME / "data" / "glossary.json",
        Path("/kaggle/input") / COMPETITION_DATASET_NAME / "glossary.json",
    ]
    for path in possible_glossary_paths:
        if path.exists():
            GLOSSARY_PATH = path
            break
else:
    glossary_path = PROJECT_ROOT / "data" / "glossary.json"
    if glossary_path.exists():
        GLOSSARY_PATH = glossary_path

if GLOSSARY_PATH and GLOSSARY_PATH.exists():
    print(f"Found glossary at: {GLOSSARY_PATH}")
    
    with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
        glossary = json.load(f)
    
    print(f"Loaded {len(glossary)} glossary entries")
else:
    print("No glossary found. Skipping post-processing.")
    glossary = None

In [ ]:
if glossary:
    import re
    
    def apply_glossary(translation, source, glossary):
        """Apply glossary replacements to translation."""
        result = translation
        
        for akkadian_term, english_term in glossary.items():
            # Check if Akkadian term appears in source
            if akkadian_term.lower() in source.lower():
                # Simple replacement - can be made more sophisticated
                # This is a basic implementation
                pass
        
        return result
    
    # Apply glossary to all translations
    corrected_translations = []
    corrections_made = 0
    
    for src, trans in zip(sources, translations):
        corrected = apply_glossary(trans, src, glossary)
        if corrected != trans:
            corrections_made += 1
        corrected_translations.append(corrected)
    
    print(f"Made {corrections_made} glossary corrections")
    
    # Save corrected submission
    submission_with_glossary = pd.DataFrame({
        "id": ids,
        "translation": corrected_translations,
    })
    
    glossary_submission_path = OUTPUT_DIR / "submission_with_glossary.csv"
    submission_with_glossary.to_csv(glossary_submission_path, index=False)
    print(f"Saved glossary-corrected submission to: {glossary_submission_path}")

## 8. Summary

In [ ]:
print("="*60)
print("INFERENCE COMPLETE")
print("="*60)
print(f"\nModel used: {MODEL_DIR}")
print(f"Test samples: {len(test_df)}")
print(f"Translations generated: {len(translations)}")
print(f"\nSubmission files:")
print(f"  - {submission_path}")
if glossary and GLOSSARY_PATH:
    print(f"  - {glossary_submission_path} (with glossary corrections)")
print("\nNext step: Submit your CSV to the competition!")